# Baseline GLM — Mitt

**Goal this week: a WORKING pipeline, not a good model.** This notebook is the first time data flows
all the way from raw `.npz` files to a trained model producing a real prediction. We're deliberately
using logistic regression (a GLM) instead of the RNN, for two reasons: it's simple enough to trust
immediately (no hidden bugs from a complex architecture), and it matches the meeting guidance to start
simple and keep interpretability front and center.

If this pipeline works end to end, swapping the GLM for the RNN later is a small, contained change,
because the hard parts (loading, labeling, windowing, evaluation) will already be tested.

**Known limitation, on purpose:** this notebook uses only Mitt, a single session, so the train/test
split has to happen at the trial level, not the session level. That's fine as a "does this work at all"
check, but it is NOT the leak-safe evaluation we'll want once multiple rats are pooled together.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score

from src.preprocessing import build_labels, segment_trials, get_sampling_rate, trial_level_split


## 1. Load raw data for Mitt

Only Mitt this week, per the plan. Loading both `bvr` (behavior/labels) and `lfp` (the actual brain
signal we'll predict from).


In [ ]:
session_dir = '../data/raw/080718_mitt'
session_name = '080718_mitt'

bvr = np.load(f'{session_dir}/{session_name}_bvr.npz', allow_pickle=True)
bvr_data = bvr['data']
bvr_keys = bvr['keys'].tolist()

lfp = np.load(f'{session_dir}/{session_name}_lfp.npz', allow_pickle=True)
lfp_data = lfp['data']

print("bvr shape:", bvr_data.shape)
print("lfp shape:", lfp_data.shape)


## 2. Extract labels and sampling rate

Using the real, implemented `preprocessing.py` functions now, not stubs. `get_sampling_rate` computes
this session's actual rate from its own `TimeBin` channel (confirmed in the multi-rat audit that this
varies by rat, so we never hardcode it). `build_labels` gives us trial indices plus both labels.


In [ ]:
fs = get_sampling_rate(bvr_data, bvr_keys)
print(f"Sampling rate: {fs:.2f} Hz")

labels = build_labels(bvr_data, bvr_keys)
print(f"Trials found: {len(labels['trial_idx'])}")
print(f"Ambiguous odor trials: {len(labels['ambiguous'])}")
print(f"InSeq/OutSeq counts: {np.bincount(labels['inseq_outseq'])}")


## 3. Cut LFP into per-trial windows

500ms windows (from `configs/baseline.yaml`), extending forward from each trial marker, since we
confirmed the marker fires at odor onset, not a later decision point.


In [ ]:
WINDOW_MS = 500

windows, kept_idx = segment_trials(lfp_data, labels['trial_idx'], WINDOW_MS, fs)
print("windows shape:", windows.shape, "-> (n_trials, n_channels, window_samples)")

# if any trials were dropped (too close to end of recording), keep labels in sync
kept_mask = np.isin(labels['trial_idx'], kept_idx)
inseq_outseq = labels['inseq_outseq'][kept_mask]
odor_id = labels['odor_id'][kept_mask]
print("Trials kept after windowing:", len(kept_idx), "(dropped:", len(labels['trial_idx']) - len(kept_idx), ")")


## 4. Turn each window into simple features

A GLM needs a flat feature vector per trial, not a raw time series (that's the RNN's job later). We use
the simplest possible summary per channel: mean and standard deviation of the voltage within the window.
This throws away timing information on purpose, the point right now is a working baseline, not a good
one.

Result: for 22 LFP channels, that's 44 features per trial (22 means + 22 std devs).


In [ ]:
def extract_simple_features(windows):
    # windows: (n_trials, n_channels, window_samples)
    means = windows.mean(axis=2)
    stds = windows.std(axis=2)
    return np.concatenate([means, stds], axis=1)  # (n_trials, n_channels * 2)

X = extract_simple_features(windows)
print("Feature matrix shape:", X.shape)


## 5. Task 1: InSeq vs OutSeq classification

Logistic regression with standardized features, evaluated with stratified k-fold cross-validation
(stratified so each fold keeps roughly the same InSeq/OutSeq ratio, important given the ~90/10
imbalance). We report both raw accuracy and balanced accuracy, and compare against the naive
"always predict the majority class" baseline, since raw accuracy alone is misleading here.


In [ ]:
y_inseq = inseq_outseq

majority_class_accuracy = max(np.mean(y_inseq == 0), np.mean(y_inseq == 1))
print(f"Naive 'always guess majority class' accuracy: {majority_class_accuracy:.3f}")
print("(Balanced accuracy for that same naive guess is always 0.5, by definition)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_acc = []
fold_bal_acc = []
for fold, (train_i, test_i) in enumerate(skf.split(X, y_inseq)):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train_i])
    X_test = scaler.transform(X[test_i])

    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, y_inseq[train_i])
    preds = clf.predict(X_test)

    acc = accuracy_score(y_inseq[test_i], preds)
    bal_acc = balanced_accuracy_score(y_inseq[test_i], preds)
    fold_acc.append(acc)
    fold_bal_acc.append(bal_acc)
    print(f"Fold {fold}: accuracy={acc:.3f}  balanced_accuracy={bal_acc:.3f}")

print()
print(f"Mean accuracy: {np.mean(fold_acc):.3f}")
print(f"Mean balanced accuracy: {np.mean(fold_bal_acc):.3f}  (chance = 0.500)")


## 6. Task 2: Odor identity, InSeq trials only

Per the meeting notes, odor classification should only be evaluated on InSeq trials. Same approach:
logistic regression (multinomial, 5 classes), stratified k-fold, compared against chance (1/5 = 20%).


In [ ]:
inseq_mask = (y_inseq == 1)
X_odor = X[inseq_mask]
y_odor = odor_id[inseq_mask]

print("InSeq trials available for odor classification:", len(y_odor))
print("Odor class counts:", np.bincount(y_odor))

skf_odor = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_acc_odor = []
fold_bal_acc_odor = []
for fold, (train_i, test_i) in enumerate(skf_odor.split(X_odor, y_odor)):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_odor[train_i])
    X_test = scaler.transform(X_odor[test_i])

    clf = LogisticRegression(max_iter=1000, multi_class='multinomial', class_weight='balanced')
    clf.fit(X_train, y_odor[train_i])
    preds = clf.predict(X_test)

    acc = accuracy_score(y_odor[test_i], preds)
    bal_acc = balanced_accuracy_score(y_odor[test_i], preds)
    fold_acc_odor.append(acc)
    fold_bal_acc_odor.append(bal_acc)
    print(f"Fold {fold}: accuracy={acc:.3f}  balanced_accuracy={bal_acc:.3f}")

print()
print(f"Mean accuracy: {np.mean(fold_acc_odor):.3f}")
print(f"Mean balanced accuracy: {np.mean(fold_bal_acc_odor):.3f}  (chance = 0.200)")


## Summary (fill in after running)

Once both tasks run, note here:
- Does InSeq/OutSeq beat chance (0.5 balanced accuracy)? By how much?
- Does odor classification beat chance (0.2 balanced accuracy)?
- Anything that looks suspicious (e.g. perfect or near-perfect accuracy is often a bug, not a good
  result, given only 292 trials and simple features)

**If this runs without errors and produces believable (not suspiciously perfect) numbers, the pipeline
is confirmed working end to end.** That's this week's goal. The next step after that is swapping this
GLM out for the MultiTaskRNN using the exact same `windows`, `inseq_outseq`, and `odor_id` arrays built
here, rather than flattened mean/std features.
